# 03 - Entrenamiento de Modelos

In [1]:
import pandas as pd
import numpy as np

from sklearn.cluster import DBSCAN
from sklearn.ensemble import IsolationForest
from sklearn.metrics import silhouette_score
from optuna import logging as optuna_logging
from optuna import samplers
from optuna import create_study

import joblib
import json

/home/miguel_blanco/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Cargar dataset preprocesado

In [2]:
df_final = pd.read_csv("../data/processed/customer_features_model.csv", index_col="CustomerID")
print(df_final.shape)
df_final.head()

(5876, 10)


,Permanencia,Compras,Canasta_Prom,Ticket_Prom,Precio_Prom,Precio_Max,Productos Distintos,Pct_Devoluciones,Comprador Local,Nomada
CustomerID,,,,,,,,,,
12347.0,0.497007,0.800347,0.996682,1.095527,-0.242354,-0.074263,0.889322,-0.803925,0,0
12348.0,0.346414,0.299455,1.408970,0.522203,-1.148191,1.162232,-0.411849,-0.803925,0,0
12349.0,1.149578,0.074223,1.094601,1.896758,0.483263,3.418458,0.963390,0.915303,0,0
12350.0,-1.055263,-1.057718,0.312880,0.265187,-0.211304,1.162232,-0.713517,-0.803925,0,0
12351.0,-1.055263,-1.057718,0.616736,0.121669,-0.695665,-0.074263,-0.548894,-0.803925,0,0


# Modelo DBSCAN

## Optimización de hiperparámetros

In [3]:
def objective_dbscan(trial):
    params_dbsc = {
        'eps' : trial.suggest_float(
            'eps', low=0.1, high=3.0
        ),

        'min_samples' : trial.suggest_int(
            'min_samples', low=2, high=50
        )
    }

    model_dbsc_trial = DBSCAN(
        **params_dbsc
    )

    labels_trial = model_dbsc_trial.fit_predict(df_final)

    n_clusters = len(set(labels_trial)) - (1 if -1 in labels_trial else 0)

    # Si no hay al menos 2 clusters reales (sin contar ruido), no se puede calcular silhouette
    if n_clusters < 2:
        return -1  # peor score posible, para que Optuna descarte esta combinación

    score_trial = silhouette_score(df_final, labels_trial)

    return score_trial

In [4]:
sampler = samplers.TPESampler(seed=42)
study_dbsc = create_study(direction='maximize', study_name='DBSC_study', sampler=sampler)

optuna_logging.set_verbosity(optuna_logging.WARNING)
study_dbsc.optimize(objective_dbscan, n_trials=20)
optuna_logging.set_verbosity(optuna_logging.INFO)

[I 2026-08-09 02:16:17,941] A new study created in memory with name: DBSC_study


In [5]:
resultados = study_dbsc.trials_dataframe()
resultados.sort_values(by="value", ascending=False, inplace=True)
resultados.drop_duplicates(subset=["value",*[col for col in resultados.columns if col.startswith('params')]], inplace=True)
resultados["value_increase"] = resultados["value"] - resultados["value"].shift(-1)
resultados[["value","value_increase"]] = resultados[["value","value_increase"]].round(4)
resultados.head(10)

,number,value,datetime_start,datetime_complete,duration,params_eps,params_min_samples,state,value_increase
14,14,0.0805,2026-08-09 02:16:31.661800,2026-08-09 02:16:32.984054,0 days 00:00:01.322254,0.882829,29,COMPLETE,0.0091
12,12,0.0714,2026-08-09 02:16:29.648573,2026-08-09 02:16:30.888260,0 days 00:00:01.239687,0.831241,22,COMPLETE,0.0444
19,19,0.0270,2026-08-09 02:16:35.843574,2026-08-09 02:16:37.133321,0 days 00:00:01.289747,0.852022,41,COMPLETE,0.0946
8,8,-0.0675,2026-08-09 02:16:24.519336,2026-08-09 02:16:25.782560,0 days 00:00:01.263224,0.982303,27,COMPLETE,0.0706
15,15,-0.1381,2026-08-09 02:16:32.984113,2026-08-09 02:16:34.167870,0 days 00:00:01.183757,0.783183,36,COMPLETE,0.0665
7,7,-0.2046,2026-08-09 02:16:23.325229,2026-08-09 02:16:24.519276,0 days 00:00:01.194047,0.627292,10,COMPLETE,0.1160
11,11,-0.3206,2026-08-09 02:16:28.074221,2026-08-09 02:16:29.648515,0 days 00:00:01.574294,0.948534,3,COMPLETE,0.0828
2,2,-0.4033,2026-08-09 02:16:19.805815,2026-08-09 02:16:21.065936,0 days 00:00:01.260121,0.552454,9,COMPLETE,0.5967
3,3,-1.0000,2026-08-09 02:16:21.065998,2026-08-09 02:16:21.202847,0 days 00:00:00.136849,0.268442,44,COMPLETE,0.0000
1,1,-1.0000,2026-08-09 02:16:18.531622,2026-08-09 02:16:19.805719,0 days 00:00:01.274097,2.222782,31,COMPLETE,0.0000


In [6]:
print(study_dbsc.best_params)
print(study_dbsc.best_value)

{'eps': 0.8828294213829456, 'min_samples': 29}
0.0805329053409582


## Entrenamiento del modelo final

In [7]:
# eps y min_samples son los hiperparámetros clave, ya optimizados
dbscan = DBSCAN(**study_dbsc.best_params)
labels_dbscan = dbscan.fit_predict(df_final)  # df_final SIN modificar

df_resultados = df_final.copy()
df_resultados['label_dbscan'] = labels_dbscan
df_resultados['anomalia_dbscan'] = np.where(labels_dbscan == -1, 1, 0)

print("Anomalías detectadas por DBSCAN:", df_resultados['anomalia_dbscan'].sum())
print("Porcentaje:", df_resultados['anomalia_dbscan'].mean() * 100, "%")

Anomalías detectadas por DBSCAN: 2295
Porcentaje: 39.0571817562968 %


**Nota:** DBSCAN es un algoritmo transductivo: `fit_predict` no expone un método `predict` independiente para asignar clientes nuevos a un cluster ya entrenado. Por eso lo que persistimos de este modelo son las etiquetas obtenidas sobre el dataset de entrenamiento, no un artefacto reutilizable para inferencia futura.

# Modelo Isolation Forest

## Optimización de hiperparámetros

In [8]:
def objective_isoforest(trial):
    params_isof = {
        'n_estimators' : trial.suggest_int(
            'n_estimators', low=50, high=300
        ),

        'max_samples' : trial.suggest_float(
            'max_samples', low=0.1, high=1.0
        ),

        'contamination' : trial.suggest_float(
            'contamination', low=0.01, high=0.4
        ),

        'max_features' : trial.suggest_float(
            'max_features', low=0.5, high=1.0
        ),
    }

    model_isof_trial = IsolationForest(
        random_state=42,
        **params_isof
    )

    labels_trial = model_isof_trial.fit_predict(df_final)

    n_clusters = len(set(labels_trial))

    # Si todos los puntos caen en una sola clase (todos normales o todos anomalia), no se puede calcular silhouette
    if n_clusters < 2:
        return -1  # peor score posible, para que Optuna descarte esta combinación

    score_trial = silhouette_score(df_final, labels_trial)

    return score_trial

In [9]:
sampler = samplers.TPESampler(seed=42)
study_isof = create_study(direction='maximize', study_name='ISOF_study', sampler=sampler)

optuna_logging.set_verbosity(optuna_logging.WARNING)
study_isof.optimize(objective_isoforest, n_trials=20)
optuna_logging.set_verbosity(optuna_logging.INFO)

[I 2026-08-09 02:16:37,649] A new study created in memory with name: ISOF_study


In [10]:
resultados = study_isof.trials_dataframe()
resultados.sort_values(by="value", ascending=False, inplace=True)
resultados.drop_duplicates(subset=["value",*[col for col in resultados.columns if col.startswith('params')]], inplace=True)
resultados["value_increase"] = resultados["value"] - resultados["value"].shift(-1)
resultados[["value","value_increase"]] = resultados[["value","value_increase"]].round(4)
resultados.head(10)

,number,value,datetime_start,datetime_complete,duration,params_contamination,params_max_features,params_max_samples,params_n_estimators,state,value_increase
12,12,0.5835,2026-08-09 02:16:55.447752,2026-08-09 02:16:57.257288,0 days 00:00:01.809536,0.013322,0.984047,0.573933,236,COMPLETE,0.0070
11,11,0.5764,2026-08-09 02:16:54.374257,2026-08-09 02:16:55.447685,0 days 00:00:01.073428,0.013751,0.990321,0.457625,54,COMPLETE,0.0056
19,19,0.5709,2026-08-09 02:17:07.273111,2026-08-09 02:17:09.061261,0 days 00:00:01.788150,0.015420,0.939807,0.658833,222,COMPLETE,0.0044
13,13,0.5665,2026-08-09 02:16:57.257354,2026-08-09 02:16:59.061989,0 days 00:00:01.804635,0.016506,0.874856,0.448173,250,COMPLETE,0.0140
2,2,0.5525,2026-08-09 02:16:40.470933,2026-08-09 02:16:42.215747,0 days 00:00:01.744814,0.018028,0.984955,0.737265,200,COMPLETE,0.0611
1,1,0.4914,2026-08-09 02:16:39.285724,2026-08-09 02:16:40.470871,0 days 00:00:01.185147,0.032653,0.933088,0.240395,89,COMPLETE,0.0812
18,18,0.4103,2026-08-09 02:17:05.691922,2026-08-09 02:17:07.273044,0 days 00:00:01.581122,0.059568,0.842752,0.492427,174,COMPLETE,0.0004
16,16,0.4098,2026-08-09 02:17:01.518463,2026-08-09 02:17:03.749914,0 days 00:00:02.231451,0.055267,0.998008,0.687703,294,COMPLETE,0.0374
3,3,0.3724,2026-08-09 02:16:42.215809,2026-08-09 02:16:44.181240,0 days 00:00:01.965431,0.080912,0.591702,0.291105,258,COMPLETE,0.0280
6,6,0.3444,2026-08-09 02:16:47.256550,2026-08-09 02:16:48.818542,0 days 00:00:01.561992,0.087873,0.757117,0.806658,164,COMPLETE,0.0062


In [11]:
print(study_isof.best_params)
print(study_isof.best_value)

{'n_estimators': 236, 'max_samples': 0.5739328498550292, 'contamination': 0.013321553045214003, 'max_features': 0.984047375706068}
0.5834507706110797


## Entrenamiento del modelo final

In [12]:
# n_estimators, max_samples, contamination y max_features son los hiperparámetros clave, ya optimizados
isoforest = IsolationForest(random_state=42, **study_isof.best_params)
labels_isof = isoforest.fit_predict(df_final)  # 1 = normal, -1 = anomalia

df_resultados['label_isof'] = labels_isof
df_resultados['anomalia_isof'] = np.where(labels_isof == -1, 1, 0)

print("Anomalías detectadas por Isolation Forest:", df_resultados['anomalia_isof'].sum())
print("Porcentaje:", df_resultados['anomalia_isof'].mean() * 100, "%")

Anomalías detectadas por Isolation Forest: 79
Porcentaje: 1.3444520081688223 %


## Guardar modelos y resultados

In [13]:
# Modelo de Isolation Forest: si tiene predict(), se puede reutilizar sobre clientes nuevos
joblib.dump(isoforest, "../models/isolation_forest.joblib")

# Mejores hiperparámetros de ambos modelos, para trazabilidad
with open("../models/best_params.json", "w") as f:
    json.dump({
        "dbscan": study_dbsc.best_params,
        "isolation_forest": study_isof.best_params,
    }, f, indent=2)

# Etiquetas crudas (label_dbscan, label_isof) y banderas de anomalia de ambos modelos
# sobre el dataset de entrenamiento, usadas en la validación
df_resultados[["label_dbscan", "anomalia_dbscan", "label_isof", "anomalia_isof"]].to_csv(
    "../data/processed/model_predictions.csv", encoding="utf-8"
)

print("Modelos y resultados guardados")

Modelos y resultados guardados
